<a href="https://colab.research.google.com/github/malyadri6761/nlp/blob/RAG/RAG_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

##**Installing Packages**

---

In [ ]:
!pip install langchain langchain-community langchain-groq
!pip install langchain-chroma langchain-huggingface
!pip install langchain-text-splitters
!pip install pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

---

##**Full Code**

---

---

**Importing Packages**

---

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

---

**1. Loading Document**

---

In [ ]:
loader = PyPDFLoader("/content/CS509_lab_test_2.pdf")
documents = loader.load()

---

**2. Split Document**

---

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)
chunks = splitter.split_documents(documents)

In [ ]:
print(len(chunks))

45


---

**3. Embeddings**

---

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

---

**4. Vector Store**

---

In [ ]:
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings
)

---

**5. Retriver**

---

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs = {'k':3}
)

---

**6. Prompt**

---

In [ ]:
prompt = ChatPromptTemplate.from_template(
    """
    Anwer the question using only the provided context.
    Context : {context}
    Question : {question}
    Answer :
    """
)

---

**7. LLM**

---


In [ ]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 0
)

---

**8.RAG Chain**

---

In [ ]:
rag_chain = (
    {
        "context": retriever | StrOutputParser(),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)



---

**9.Query**

---

In [ ]:
from langchain_core.runnables import RunnableLambda

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

question = "Bfs assginement?"
answer = rag_chain.invoke(question)
print(answer)

**BFS assignment** refers to the process of labeling every vertex in the graph with its **BFS level** (i.e., the shortest‑path distance in edges from a chosen root vertex).  
- **Level 0** contains only the root.  
- **Level 1** contains all vertices adjacent to the root.  
- **Level 2** contains all vertices that are two edges away from the root, and so on.  

Within a single level, vertices are ordered by a *consistent tie‑breaking rule* (for example, ascending vertex ID) because a plain BFS traversal does not impose an order on vertices that are discovered at the same depth. This ordered list of vertices is then used to drive the greedy coloring algorithm.
